In [1]:
import re
import ast
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import joblib

RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")
REPORTS = Path("../Reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW.resolve())
print("PROCESSED:", PROCESSED.resolve())
print("REPORTS:", REPORTS.resolve())

RAW: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Raw
PROCESSED: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Processed
REPORTS: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Reports


In [2]:
def to_list_safe(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    if isinstance(x, (tuple, set)):
        return [str(i).strip() for i in x if str(i).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s or s.lower() == "nan":
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return [str(i).strip() for i in parsed if str(i).strip()]
            except Exception:
                pass
        if "|" in s:
            return [i.strip() for i in s.split("|") if i.strip()]
        if "," in s:
            return [i.strip() for i in s.split(",") if i.strip()]
        return [s]
    return [str(x).strip()]


def normalize_token(s):
    return str(s).strip().lower()


def normalize_list(x):
    return [normalize_token(i) for i in to_list_safe(x) if normalize_token(i)]


def overlap_items(candidate_values, user_values):
    cand_raw = to_list_safe(candidate_values)
    user_norm = set(normalize_list(user_values))

    out = []
    seen = set()
    for item in cand_raw:
        norm = normalize_token(item)
        if norm in user_norm and norm not in seen:
            out.append(str(item).strip())
            seen.add(norm)
    return out


def overlap_score(candidate_values, user_values):
    user_norm = set(normalize_list(user_values))
    if not user_norm:
        return 0.0
    cand_norm = set(normalize_list(candidate_values))
    hit = len(cand_norm.intersection(user_norm))
    return hit / len(user_norm)


def coalesce_list_columns(df, base_name):
    candidates = [
        c for c in [
            base_name,
            f"{base_name}_x",
            f"{base_name}_y",
            f"{base_name}_title",
            f"{base_name}_trait",
        ]
        if c in df.columns
    ]

    if not candidates:
        df[base_name] = [[] for _ in range(len(df))]
        return df

    def combine_row(row):
        merged = []
        seen = set()
        for c in candidates:
            for item in to_list_safe(row[c]):
                norm = normalize_token(item)
                if norm and norm not in seen:
                    merged.append(str(item).strip())
                    seen.add(norm)
        return merged

    df[base_name] = df.apply(combine_row, axis=1)

    for c in candidates:
        if c != base_name and c in df.columns:
            df.drop(columns=c, inplace=True)

    return df


def safe_minmax(values, full_index):
    s = pd.Series(values, index=full_index if not isinstance(values, pd.Series) else values.index)
    s = s.reindex(full_index).astype(float)

    if len(s) == 0:
        return pd.Series(dtype=float)

    arr = s.values
    if np.all(np.isnan(arr)):
        return pd.Series(0.0, index=full_index)

    mn = np.nanmin(arr)
    mx = np.nanmax(arr)

    if mx == mn:
        return pd.Series(0.0, index=full_index)

    arr = np.nan_to_num(arr, nan=np.nanmean(arr))
    norm = (arr - mn) / (mx - mn + 1e-9)
    return pd.Series(norm, index=full_index)

In [3]:
meta_path = RAW / "games_detailed_info.csv"
df_meta = pd.read_csv(meta_path, low_memory=False)

df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce")
df_meta = df_meta.dropna(subset=["id"]).copy()
df_meta["id"] = df_meta["id"].astype(int)
df_meta = df_meta.drop_duplicates(subset=["id"]).copy()

games = df_meta.copy()

possible_name_cols = ["name", "primary", "title", "game_name", "Name"]
detected_name_col = None
for c in possible_name_cols:
    if c in games.columns:
        detected_name_col = c
        break

if detected_name_col is None:
    raise ValueError(f"Could not detect title column. Available columns: {list(games.columns)}")

NAME_COL = detected_name_col
MECH_COL = "boardgamemechanic"
THEME_COL = "boardgamecategory"
FAMILY_COL = "boardgamefamily"
PUB_COL = "boardgamepublisher"

games["name"] = games[NAME_COL].astype(str)

if "averageweight" in games.columns:
    games["complexity"] = pd.to_numeric(games["averageweight"], errors="coerce")
else:
    games["complexity"] = np.nan

if "usersrated" in games.columns:
    games["num_votes"] = pd.to_numeric(games["usersrated"], errors="coerce")
else:
    games["num_votes"] = np.nan

if "bayesaverage" in games.columns:
    games["avg_rating"] = pd.to_numeric(games["bayesaverage"], errors="coerce")
elif "average" in games.columns:
    games["avg_rating"] = pd.to_numeric(games["average"], errors="coerce")
else:
    games["avg_rating"] = np.nan

if "yearpublished" in games.columns:
    games["yearpublished"] = pd.to_numeric(games["yearpublished"], errors="coerce")
else:
    games["yearpublished"] = np.nan

TRAIT_COLUMNS = [MECH_COL, THEME_COL, FAMILY_COL, PUB_COL]
for col in TRAIT_COLUMNS:
    if col in games.columns:
        games[col] = games[col].apply(to_list_safe)

df_meta_clean = games.set_index("id").copy()

print("Detected NAME_COL:", NAME_COL)
display(games[["id", "name", "avg_rating", "complexity", "num_votes"]].head())

Detected NAME_COL: primary


,id,name,avg_rating,complexity,num_votes
0,30549,Pandemic,7.48669,2.4063,109006
1,822,Carcassonne,7.30857,1.9057,108776
2,13,Catan,6.96965,2.3130,108064
3,68448,7 Wonders,7.63355,2.3264,90021
4,36218,Dominion,7.49912,2.3542,81582


In [4]:
def norm_token(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def to_tokens(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, list):
        return [norm_token(t) for t in x if str(t).strip()]

    s = str(x).strip()
    if not s:
        return []

    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")

    if "|" in s:
        parts = s.split("|")
    elif "," in s:
        parts = s.split(",")
    else:
        parts = [s]

    return [norm_token(p) for p in parts if str(p).strip()]

def get_tokens_series(df, col):
    if col not in df.columns:
        return pd.Series([[]] * len(df), index=df.index)
    return df[col].apply(to_tokens)

mech_tokens = get_tokens_series(df_meta_clean, MECH_COL)
cat_tokens = get_tokens_series(df_meta_clean, THEME_COL)
fam_tokens = get_tokens_series(df_meta_clean, FAMILY_COL)
pub_tokens = get_tokens_series(df_meta_clean, PUB_COL)

In [5]:
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")

meta_ids = df_meta_clean.index.to_numpy()
id_to_idx = {int(gid): i for i, gid in enumerate(meta_ids)}
idx_to_id = {i: int(gid) for i, gid in enumerate(meta_ids)}

print("Similarity matrices loaded.")
print("Universe size:", len(meta_ids))

Similarity matrices loaded.
Universe size: 21631


In [6]:
sent_path = PROCESSED / "sentiment_summary.csv"

if sent_path.exists():
    sent_summary = pd.read_csv(sent_path, index_col="id")
    sent_summary.index = pd.to_numeric(sent_summary.index, errors="coerce")
    sent_summary = sent_summary[~pd.isna(sent_summary.index)].copy()
    sent_summary.index = sent_summary.index.astype(int)

    if "avg_sentiment_score" in sent_summary.columns:
        sent_mean = float(sent_summary["avg_sentiment_score"].mean())
        games["sentiment_score"] = games["id"].map(sent_summary["avg_sentiment_score"]).fillna(sent_mean)
    else:
        sent_summary = None
        sent_mean = 0.0
        games["sentiment_score"] = 0.0
else:
    sent_summary = None
    sent_mean = 0.0
    games["sentiment_score"] = 0.0

print("Sentiment ready:", sent_summary is not None)
print("sent_mean:", sent_mean)

Sentiment ready: True
sent_mean: 0.9701084770133322


In [7]:
candidate_rating_files = [
    RAW / "large_user_ratings.csv",
    RAW / "large_bgg-15m-reviews.csv",
    RAW / "large_bgg-19m-reviews.csv",
    RAW / "large_bgg-26m-reviews.csv",
]

ratings_path = None
for p in candidate_rating_files:
    if p.exists():
        ratings_path = p
        break

if ratings_path is None:
    raise FileNotFoundError("Could not find ratings/reviews file in Dataset/Raw")

print("Using ratings file:", ratings_path.name)

ratings_df = pd.read_csv(ratings_path, low_memory=False)
print("Raw ratings shape:", ratings_df.shape)
print("Columns:", list(ratings_df.columns))

Using ratings file: large_user_ratings.csv
Raw ratings shape: (18942215, 3)
Columns: ['BGGId', 'Rating', 'Username']


In [8]:
def detect_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

USER_COL = detect_first_existing(ratings_df.columns, ["Username", "username", "user", "User", "reviewer"])
ITEM_COL = detect_first_existing(ratings_df.columns, ["BGGId", "bgg_id", "game_id", "id", "item_id"])
RATING_COL = detect_first_existing(ratings_df.columns, ["Rating", "rating", "score"])

print("Detected USER_COL:", USER_COL)
print("Detected ITEM_COL:", ITEM_COL)
print("Detected RATING_COL:", RATING_COL)

if USER_COL is None or ITEM_COL is None:
    raise ValueError("Could not detect user/item columns in ratings file.")

ratings_df = ratings_df.dropna(subset=[USER_COL, ITEM_COL]).copy()
ratings_df[ITEM_COL] = pd.to_numeric(ratings_df[ITEM_COL], errors="coerce")
ratings_df = ratings_df.dropna(subset=[ITEM_COL]).copy()
ratings_df[ITEM_COL] = ratings_df[ITEM_COL].astype(int)

if RATING_COL is not None:
    ratings_df[RATING_COL] = pd.to_numeric(ratings_df[RATING_COL], errors="coerce")
else:
    ratings_df["__implicit_rating__"] = 1.0
    RATING_COL = "__implicit_rating__"

ratings_df = ratings_df[ratings_df[ITEM_COL].isin(meta_ids)].copy()

print("Filtered ratings shape:", ratings_df.shape)
display(ratings_df[[USER_COL, ITEM_COL, RATING_COL]].head())

Detected USER_COL: Username
Detected ITEM_COL: BGGId
Detected RATING_COL: Rating
Filtered ratings shape: (18744924, 3)


,Username,BGGId,Rating
75,Narfbuster,193500,5.0
76,Methrin,193500,5.0
77,Evabelle,193500,5.0
78,ngcx6611,193500,5.0
79,bmillerbwm,193500,5.0


In [9]:
pop_counts = ratings_df.groupby(ITEM_COL).size()
pop_counts = pop_counts.reindex(meta_ids).fillna(0).astype(int)

pop_norm = (pop_counts - pop_counts.min()) / (pop_counts.max() - pop_counts.min() + 1e-9)
pop_norm = pop_norm.fillna(0.0)

games["popularity_norm"] = games["id"].map(pop_norm).fillna(0.0)

print("Popularity signal ready.")

Popularity signal ready.


In [10]:
def topk_from_sim(seed_ids, sim_csr, k=1200, strategy="mean"):
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        for c, v in zip(row.indices, row.data):
            gid = idx_to_id[c]
            if strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] /= max(counts[gid], 1)

    for sid in seed_ids:
        scores.pop(int(sid), None)

    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)

In [11]:
df_names = df_meta_clean[["name"]].copy()
df_names["name_norm"] = df_names["name"].fillna("").apply(norm_token)

def search_titles(query, topn=10):
    q = norm_token(query)
    if not q:
        return pd.DataFrame(columns=["id", "name"])

    exact = df_names[df_names["name_norm"] == q].reset_index()[["id", "name"]]
    if len(exact) > 0:
        return exact.head(topn)

    cand = df_names[df_names["name_norm"].str.contains(re.escape(q), na=False)].reset_index()[["id", "name"]]

    if "num_votes" in games.columns:
        cand = cand.merge(games[["id", "num_votes"]], on="id", how="left")
        cand["num_votes"] = pd.to_numeric(cand["num_votes"], errors="coerce").fillna(0)
        cand = cand.sort_values("num_votes", ascending=False).drop(columns=["num_votes"])

    return cand.head(topn)

def resolve_titles_to_ids(titles, pick="auto"):
    seed_ids = []
    candidates_info = []

    for t in titles:
        matches = search_titles(t, topn=10)

        if len(matches) == 0:
            candidates_info.append((t, []))
            continue

        ids = matches["id"].tolist()
        candidates_info.append((t, ids))

        if pick == "all":
            seed_ids.extend(ids)
        else:
            seed_ids.append(int(ids[0]))

    seen = set()
    uniq = []
    for x in seed_ids:
        if int(x) not in seen:
            uniq.append(int(x))
            seen.add(int(x))

    return uniq, candidates_info

display(search_titles("Splendor", topn=5))

,id,name
0,148228,Splendor


In [12]:
def build_type_candidates(
    games_df,
    wanted_categories=None,
    wanted_mechanics=None,
    wanted_families=None,
    wanted_publishers=None,
    min_rating=None,
    max_weight=None,
    difficulty_label=None
):
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []
    wanted_families = wanted_families or []
    wanted_publishers = wanted_publishers or []

    df = games_df.copy()

    if THEME_COL in df.columns:
        df["matched_categories"] = df[THEME_COL].apply(lambda x: overlap_items(x, wanted_categories))
        df["score_theme"] = df[THEME_COL].apply(lambda x: overlap_score(x, wanted_categories))
    else:
        df["matched_categories"] = [[] for _ in range(len(df))]
        df["score_theme"] = 0.0

    if MECH_COL in df.columns:
        df["matched_mechanics"] = df[MECH_COL].apply(lambda x: overlap_items(x, wanted_mechanics))
        df["score_mech"] = df[MECH_COL].apply(lambda x: overlap_score(x, wanted_mechanics))
    else:
        df["matched_mechanics"] = [[] for _ in range(len(df))]
        df["score_mech"] = 0.0

    if FAMILY_COL in df.columns:
        df["matched_families"] = df[FAMILY_COL].apply(lambda x: overlap_items(x, wanted_families))
        df["score_family"] = df[FAMILY_COL].apply(lambda x: overlap_score(x, wanted_families))
    else:
        df["matched_families"] = [[] for _ in range(len(df))]
        df["score_family"] = 0.0

    if PUB_COL in df.columns:
        df["matched_publishers"] = df[PUB_COL].apply(lambda x: overlap_items(x, wanted_publishers))
        df["score_publisher"] = df[PUB_COL].apply(lambda x: overlap_score(x, wanted_publishers))
    else:
        df["matched_publishers"] = [[] for _ in range(len(df))]
        df["score_publisher"] = 0.0

    df["score_trait"] = (
        0.40 * df["score_theme"] +
        0.35 * df["score_mech"] +
        0.15 * df["score_family"] +
        0.10 * df["score_publisher"]
    )

    if difficulty_label is not None:
        diff = str(difficulty_label).strip().lower()
        if "complexity" in df.columns:
            s = pd.to_numeric(df["complexity"], errors="coerce")
            mask = pd.Series(True, index=df.index)

            if diff == "low":
                mask &= (s <= 2.0)
            elif diff == "medium":
                mask &= (s > 2.0) & (s <= 3.0)
            elif diff == "high":
                mask &= (s > 3.0)

            df = df[mask.fillna(False)].copy()

    if min_rating is not None and "avg_rating" in df.columns:
        df = df[pd.to_numeric(df["avg_rating"], errors="coerce") >= min_rating].copy()

    if max_weight is not None and "complexity" in df.columns:
        df = df[pd.to_numeric(df["complexity"], errors="coerce") <= max_weight].copy()

    trait_cols = ["score_theme", "score_mech", "score_family", "score_publisher"]
    df = df[df[trait_cols].sum(axis=1) > 0].copy()

    return df

In [13]:
def recommend_by_titles(
    query_titles,
    top_n=100,
    w_text=0.35,
    w_emb=0.35,
    w_sent=0.10,
    w_pop=0.20,
    candidate_k=1200,
    pick="auto"
):
    seed_ids, candidates_info = resolve_titles_to_ids(query_titles, pick=pick)
    seed_ids = [sid for sid in seed_ids if sid in id_to_idx]

    if len(seed_ids) == 0:
        return pd.DataFrame(columns=["id", "name", "score_like"])

    desc_s = topk_from_sim(seed_ids, desc_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    cont_s = topk_from_sim(seed_ids, content_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    emb_s = topk_from_sim(seed_ids, emb_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)

    text_norm = safe_minmax(0.5 * desc_s + 0.5 * cont_s, meta_ids)
    emb_norm = safe_minmax(emb_s, meta_ids)

    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids).fillna(sent_mean)
        sent_norm = safe_minmax(sent, meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    pop = pop_norm.reindex(meta_ids).fillna(0.0)

    score_like = w_text * text_norm + w_emb * emb_norm + w_sent * sent_norm + w_pop * pop

    for sid in seed_ids:
        if sid in score_like.index:
            score_like.loc[sid] = -1

    rec_ids = score_like.sort_values(ascending=False).head(top_n).index.astype(int).tolist()

    seed_mech = set().union(*[set(mech_tokens.loc[sid]) for sid in seed_ids if sid in mech_tokens.index]) if len(seed_ids) > 0 else set()
    seed_cat = set().union(*[set(cat_tokens.loc[sid]) for sid in seed_ids if sid in cat_tokens.index]) if len(seed_ids) > 0 else set()
    seed_pub = set().union(*[set(pub_tokens.loc[sid]) for sid in seed_ids if sid in pub_tokens.index]) if len(seed_ids) > 0 else set()

    out = []
    for gid in rec_ids:
        m = set(mech_tokens.loc[gid]) if gid in mech_tokens.index else set()
        c = set(cat_tokens.loc[gid]) if gid in cat_tokens.index else set()
        p = set(pub_tokens.loc[gid]) if gid in pub_tokens.index else set()

        out.append({
            "id": gid,
            "name": df_meta_clean.loc[gid, "name"] if gid in df_meta_clean.index else "",
            "score_like": float(score_like.loc[gid]),
            "matched_mechanics": sorted(list(m & seed_mech))[:8],
            "matched_categories": sorted(list(c & seed_cat))[:8],
            "matched_publishers": sorted(list(p & seed_pub))[:6],
        })

    return pd.DataFrame(out).sort_values("score_like", ascending=False).reset_index(drop=True)

In [14]:
TRAIT_EXPANSIONS = {
    "strategy": {
        "categories": ["strategy", "economic", "puzzle", "territory building", "city building"],
        "mechanics": ["hand management", "tile placement", "area majority / influence", "set collection"]
    },
    "family": {
        "categories": ["family", "children's game", "party game", "card game"],
        "mechanics": ["tile placement", "set collection", "pattern building"]
    },
    "horror": {
        "categories": ["horror"],
        "mechanics": ["cooperative game", "hidden movement"]
    },
    "mystery": {
        "categories": ["mystery", "deduction"],
        "mechanics": ["deduction", "hidden roles", "memory"]
    },
    "fun": {
        "categories": ["party game", "humor"],
        "mechanics": ["push your luck", "real-time", "voting"]
    },
    "co-op": {
        "categories": [],
        "mechanics": ["cooperative game"]
    },
    "coop": {
        "categories": [],
        "mechanics": ["cooperative game"]
    }
}

def expand_traits(wanted_categories=None, wanted_mechanics=None):
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []

    cat_out = list(wanted_categories)
    mech_out = list(wanted_mechanics)

    for item in wanted_categories:
        key = str(item).strip().lower()
        if key in TRAIT_EXPANSIONS:
            cat_out.extend(TRAIT_EXPANSIONS[key].get("categories", []))
            mech_out.extend(TRAIT_EXPANSIONS[key].get("mechanics", []))

    cat_out = list(dict.fromkeys([str(x).strip() for x in cat_out if str(x).strip()]))
    mech_out = list(dict.fromkeys([str(x).strip() for x in mech_out if str(x).strip()]))

    return cat_out, mech_out

In [15]:
def make_reason(row):
    reasons = []

    cats = to_list_safe(row.get("matched_categories", []))
    mechs = to_list_safe(row.get("matched_mechanics", []))
    fams = to_list_safe(row.get("matched_families", []))
    pubs = to_list_safe(row.get("matched_publishers", []))

    if len(cats) > 0:
        reasons.append("Categories: " + ", ".join(cats))
    if len(mechs) > 0:
        reasons.append("Mechanics: " + ", ".join(mechs))
    if len(fams) > 0:
        reasons.append("Families: " + ", ".join(fams))
    if len(pubs) > 0:
        reasons.append("Publishers: " + ", ".join(pubs))

    return " | ".join(reasons) if reasons else "No explicit trait match recorded"


def discover(
    query_titles=None,
    wanted_categories=None,
    wanted_mechanics=None,
    wanted_families=None,
    wanted_publishers=None,
    difficulty_label=None,
    top_n=20,
    alpha_title=0.65,
    alpha_trait=0.35,
    exclude_query_titles=True
):
    query_titles = query_titles or []
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []
    wanted_families = wanted_families or []
    wanted_publishers = wanted_publishers or []

    wanted_categories, wanted_mechanics = expand_traits(
        wanted_categories=wanted_categories,
        wanted_mechanics=wanted_mechanics
    )

    has_title = len(query_titles) > 0
    has_trait = any([
        len(wanted_categories) > 0,
        len(wanted_mechanics) > 0,
        len(wanted_families) > 0,
        len(wanted_publishers) > 0,
        difficulty_label is not None
    ])

    if not has_title and not has_trait:
        raise ValueError("Provide at least one title or one trait.")

    title_df = None
    if has_title:
        title_df = recommend_by_titles(
            query_titles=query_titles,
            top_n=max(top_n * 5, 100)
        ).copy()

    trait_df = None
    if has_trait:
        trait_df = build_type_candidates(
            games_df=games,
            wanted_categories=wanted_categories,
            wanted_mechanics=wanted_mechanics,
            wanted_families=wanted_families,
            wanted_publishers=wanted_publishers,
            difficulty_label=difficulty_label
        ).copy()

        keep_cols = [
            "id", "name",
            "matched_categories", "matched_mechanics",
            "matched_families", "matched_publishers",
            "score_theme", "score_mech", "score_family",
            "score_publisher", "score_trait"
        ]
        keep_cols = [c for c in keep_cols if c in trait_df.columns]
        trait_df = trait_df[keep_cols].copy()

    meta_cols = ["id", "name", "avg_rating", "complexity", "yearpublished", "sentiment_score", "num_votes"]
    meta_cols = [c for c in meta_cols if c in games.columns]

    if has_title and not has_trait:
        out = title_df.copy()
        out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")
        out["score_trait"] = 0.0
        out["final_score"] = out["score_like"]
        out["reason"] = out.apply(make_reason, axis=1)
        return out.sort_values(["final_score", "avg_rating", "num_votes"], ascending=[False, False, False]).head(top_n).reset_index(drop=True)

    if has_trait and not has_title:
        out = trait_df.copy()
        out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")
        out["score_like"] = 0.0

        out["rating_norm"] = safe_minmax(pd.to_numeric(out["avg_rating"], errors="coerce").fillna(0.0), out.index)
        out["votes_norm"] = safe_minmax(np.log1p(pd.to_numeric(out["num_votes"], errors="coerce").fillna(0.0)), out.index)
        out["sent_norm"] = safe_minmax(pd.to_numeric(out["sentiment_score"], errors="coerce").fillna(0.0), out.index)

        out["final_score"] = (
            0.70 * out["score_trait"] +
            0.15 * out["rating_norm"] +
            0.10 * out["votes_norm"] +
            0.05 * out["sent_norm"]
        )

        out["reason"] = out.apply(make_reason, axis=1)
        return out.sort_values(["final_score", "score_trait", "avg_rating", "num_votes"], ascending=[False, False, False, False]).head(top_n).reset_index(drop=True)

    out = title_df.merge(
        trait_df,
        on=["id", "name"],
        how="outer",
        suffixes=("_title", "_trait")
    ).copy()

    if "score_like" not in out.columns:
        out["score_like"] = 0.0
    out["score_like"] = out["score_like"].fillna(0.0)

    for col in ["matched_categories", "matched_mechanics", "matched_families", "matched_publishers"]:
        out = coalesce_list_columns(out, col)

    for col in ["score_theme", "score_mech", "score_family", "score_publisher", "score_trait"]:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].fillna(0.0)

    out["score_trait"] = (
        0.40 * out["score_theme"] +
        0.35 * out["score_mech"] +
        0.15 * out["score_family"] +
        0.10 * out["score_publisher"]
    )

    out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")

    if exclude_query_titles:
        query_titles_norm = {str(t).strip().lower() for t in query_titles}
        out = out[~out["name"].astype(str).str.strip().str.lower().isin(query_titles_norm)].copy()

    out["rating_norm"] = safe_minmax(pd.to_numeric(out["avg_rating"], errors="coerce").fillna(0.0), out.index)
    out["votes_norm"] = safe_minmax(np.log1p(pd.to_numeric(out["num_votes"], errors="coerce").fillna(0.0)), out.index)
    out["sent_norm"] = safe_minmax(pd.to_numeric(out["sentiment_score"], errors="coerce").fillna(0.0), out.index)

    out["final_score"] = (
        alpha_title * out["score_like"] +
        alpha_trait * out["score_trait"] +
        0.08 * out["rating_norm"] +
        0.04 * out["votes_norm"] +
        0.03 * out["sent_norm"]
    )

    out["reason"] = out.apply(make_reason, axis=1)

    return out.sort_values(
        ["final_score", "score_like", "score_trait", "avg_rating", "num_votes"],
        ascending=[False, False, False, False, False]
    ).head(top_n).reset_index(drop=True)

In [16]:
def recall_at_k(recommended_ids, holdout_ids, k=10):
    rec_k = list(recommended_ids)[:k]
    holdout = set(holdout_ids)
    if len(holdout) == 0:
        return np.nan
    hits = sum(1 for x in rec_k if x in holdout)
    return hits / len(holdout)

def ndcg_at_k(recommended_ids, holdout_ids, k=10):
    rec_k = list(recommended_ids)[:k]
    holdout = set(holdout_ids)
    if len(holdout) == 0:
        return np.nan

    dcg = 0.0
    for i, item in enumerate(rec_k):
        if item in holdout:
            dcg += 1.0 / math.log2(i + 2)

    ideal_hits = min(len(holdout), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    if idcg == 0:
        return 0.0
    return dcg / idcg

In [17]:
user_hist = ratings_df.groupby(USER_COL)[ITEM_COL].apply(list)

# remove duplicates while preserving order
def dedupe_keep_order(lst):
    seen = set()
    out = []
    for x in lst:
        if x not in seen:
            out.append(int(x))
            seen.add(int(x))
    return out

user_hist = user_hist.apply(dedupe_keep_order)

hist_len = user_hist.apply(len)
eligible_users = user_hist[hist_len >= 6].index.tolist()

print("Total users:", len(user_hist))
print("Eligible users (>=6 items):", len(eligible_users))

Total users: 410471
Eligible users (>=6 items): 259737


In [18]:
def infer_traits_from_seed_ids(seed_ids, top_n_each=3):
    cat_counter = defaultdict(int)
    mech_counter = defaultdict(int)

    for sid in seed_ids:
        if sid in cat_tokens.index:
            for c in cat_tokens.loc[sid]:
                cat_counter[c] += 1
        if sid in mech_tokens.index:
            for m in mech_tokens.loc[sid]:
                mech_counter[m] += 1

    top_cats = [k for k, _ in sorted(cat_counter.items(), key=lambda x: (-x[1], x[0]))[:top_n_each]]
    top_mechs = [k for k, _ in sorted(mech_counter.items(), key=lambda x: (-x[1], x[0]))[:top_n_each]]

    return top_cats, top_mechs

In [19]:
def evaluate_one_user(user_id, seed_size=3, holdout_size=3, k_list=(5, 10, 20)):
    items = user_hist.loc[user_id]
    if len(items) < seed_size + holdout_size:
        return None

    seed_ids = items[:seed_size]
    holdout_ids = items[seed_size:seed_size + holdout_size]

    seed_titles = []
    for sid in seed_ids:
        if sid in df_meta_clean.index:
            seed_titles.append(df_meta_clean.loc[sid, "name"])

    wanted_categories, wanted_mechanics = infer_traits_from_seed_ids(seed_ids, top_n_each=2)

    # title-only
    title_res = discover(
        query_titles=seed_titles,
        top_n=max(k_list)
    )
    title_ids = title_res["id"].tolist() if "id" in title_res.columns else []

    # trait-only
    trait_res = discover(
        wanted_categories=wanted_categories,
        wanted_mechanics=wanted_mechanics,
        top_n=max(k_list)
    )
    trait_ids = trait_res["id"].tolist() if "id" in trait_res.columns else []

    # combined
    combined_res = discover(
        query_titles=seed_titles,
        wanted_categories=wanted_categories,
        wanted_mechanics=wanted_mechanics,
        top_n=max(k_list)
    )
    combined_ids = combined_res["id"].tolist() if "id" in combined_res.columns else []

    row = {
        "user_id": user_id,
        "seed_ids": seed_ids,
        "holdout_ids": holdout_ids,
        "seed_titles": seed_titles,
        "wanted_categories": wanted_categories,
        "wanted_mechanics": wanted_mechanics,
    }

    for k in k_list:
        row[f"title_recall@{k}"] = recall_at_k(title_ids, holdout_ids, k=k)
        row[f"title_ndcg@{k}"] = ndcg_at_k(title_ids, holdout_ids, k=k)

        row[f"trait_recall@{k}"] = recall_at_k(trait_ids, holdout_ids, k=k)
        row[f"trait_ndcg@{k}"] = ndcg_at_k(trait_ids, holdout_ids, k=k)

        row[f"combined_recall@{k}"] = recall_at_k(combined_ids, holdout_ids, k=k)
        row[f"combined_ndcg@{k}"] = ndcg_at_k(combined_ids, holdout_ids, k=k)

    return row

In [20]:
sample_user = eligible_users[0]
sample_eval = evaluate_one_user(sample_user, seed_size=3, holdout_size=3, k_list=(5, 10, 20))

print("Sample user:", sample_user)
for k, v in sample_eval.items():
    if not isinstance(v, list):
        print(k, ":", v)

Sample user:  beastvol
user_id :  beastvol
title_recall@5 : 0.0
title_ndcg@5 : 0.0
trait_recall@5 : 0.0
trait_ndcg@5 : 0.0
combined_recall@5 : 0.0
combined_ndcg@5 : 0.0
title_recall@10 : 0.0
title_ndcg@10 : 0.0
trait_recall@10 : 0.0
trait_ndcg@10 : 0.0
combined_recall@10 : 0.0
combined_ndcg@10 : 0.0
title_recall@20 : 0.0
title_ndcg@20 : 0.0
trait_recall@20 : 0.0
trait_ndcg@20 : 0.0
combined_recall@20 : 0.0
combined_ndcg@20 : 0.0


In [21]:
MAX_USERS = 300
SEED_SIZE = 3
HOLDOUT_SIZE = 3
K_LIST = (5, 10, 20)

eval_users = eligible_users[:MAX_USERS]

rows = []
for i, uid in enumerate(eval_users, start=1):
    result = evaluate_one_user(uid, seed_size=SEED_SIZE, holdout_size=HOLDOUT_SIZE, k_list=K_LIST)
    if result is not None:
        rows.append(result)

    if i % 25 == 0:
        print(f"Processed {i}/{len(eval_users)} users ...")

eval_df = pd.DataFrame(rows)
print("Evaluation rows:", eval_df.shape)
display(eval_df.head(3))

Processed 25/300 users ...
Processed 50/300 users ...
Processed 75/300 users ...
Processed 100/300 users ...
Processed 125/300 users ...
Processed 150/300 users ...
Processed 175/300 users ...
Processed 200/300 users ...
Processed 225/300 users ...
Processed 250/300 users ...
Processed 275/300 users ...
Processed 300/300 users ...
Evaluation rows: (300, 24)


,user_id,seed_ids,holdout_ids,seed_titles,wanted_categories,wanted_mechanics,title_recall@5,title_ndcg@5,trait_recall@5,trait_ndcg@5,...,trait_recall@10,trait_ndcg@10,combined_recall@10,combined_ndcg@10,title_recall@20,title_ndcg@20,trait_recall@20,trait_ndcg@20,combined_recall@20,combined_ndcg@20
0,beastvol,"[9209, 13, 3076]","[118, 823, 278]","[Ticket to Ride, Catan, Puerto Rico]","[economic, city building]","[end game bonuses, network and route building]",0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
1,mycroft,"[15987, 278, 13]","[10547, 823, 5]","[Arkham Horror, Catan Card Game, Catan]","[adventure, card game]","[dice rolling, hand management]",0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
2,-=Yod@=-,"[257707, 264295, 38949]","[210179, 822, 13]",[Unlock!: Escape Adventures – In Pursuit of Ca...,"[card game, children's game]","[area majority / influence, cooperative game]",0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.333333,0.108581,0.0,0.0,0.0,0.0


In [22]:
def summarize_model(eval_df, model_name, k_list=(5, 10, 20)):
    row = {"model": model_name}
    for k in k_list:
        row[f"Recall@{k}"] = eval_df[f"{model_name}_recall@{k}"].mean()
        row[f"NDCG@{k}"] = eval_df[f"{model_name}_ndcg@{k}"].mean()
    return row

summary_rows = [
    summarize_model(eval_df, "title", K_LIST),
    summarize_model(eval_df, "trait", K_LIST),
    summarize_model(eval_df, "combined", K_LIST),
]

summary_df = pd.DataFrame(summary_rows)

for c in summary_df.columns:
    if c != "model":
        summary_df[c] = pd.to_numeric(summary_df[c], errors="coerce").round(4)

display(summary_df)

,model,Recall@5,NDCG@5,Recall@10,NDCG@10,Recall@20,NDCG@20
0,title,0.1144,0.0977,0.2033,0.1377,0.3011,0.1726
1,trait,0.0144,0.0137,0.0189,0.0156,0.0367,0.0219
2,combined,0.0967,0.0846,0.1789,0.1217,0.2822,0.1588


In [23]:
best_by_metric = {}

for metric in [c for c in summary_df.columns if c != "model"]:
    best_row = summary_df.sort_values(metric, ascending=False).iloc[0]
    best_by_metric[metric] = {
        "best_model": best_row["model"],
        "score": best_row[metric]
    }

best_metric_df = pd.DataFrame(best_by_metric).T.reset_index().rename(columns={"index": "metric"})
display(best_metric_df)

,metric,best_model,score
0,Recall@5,title,0.1144
1,NDCG@5,title,0.0977
2,Recall@10,title,0.2033
3,NDCG@10,title,0.1377
4,Recall@20,title,0.3011
5,NDCG@20,title,0.1726


In [24]:
summary_df.to_csv(REPORTS / "notebook09_summary_metrics.csv", index=False)
eval_df.to_csv(REPORTS / "notebook09_user_level_results.csv", index=False)

print("Saved:")
print(REPORTS / "notebook09_summary_metrics.csv")
print(REPORTS / "notebook09_user_level_results.csv")

Saved:
..\Reports\notebook09_summary_metrics.csv
..\Reports\notebook09_user_level_results.csv


In [25]:
case_title_titles = ["Splendor", "Azul"]

case_title = discover(
    query_titles=case_title_titles,
    top_n=10
)

print("=== Case Study: Title-only ===")
print("Input titles:", case_title_titles)
display(case_title[["id", "name", "score_like", "final_score", "avg_rating", "num_votes", "reason"]].head(10))

=== Case Study: Title-only ===
Input titles: ['Splendor', 'Azul']


,id,name,score_like,final_score,avg_rating,num_votes,reason
0,822,Carcassonne,0.630833,0.630833,7.30857,108776,Mechanics: tile placement | Publishers: filoso...
1,13,Catan,0.611436,0.611436,6.96965,108064,Categories: economic | Mechanics: race | Publi...
2,30549,Pandemic,0.592768,0.592768,7.48669,109006,Mechanics: set collection | Publishers: asmode...
3,9209,Ticket to Ride,0.586966,0.586966,7.30509,76207,"Mechanics: card drafting, contracts, end game ..."
4,68448,7 Wonders,0.576293,0.576293,7.63355,90021,"Categories: card game, economic | Mechanics: s..."
5,178900,Codenames,0.534498,0.534498,7.50765,74456,"Categories: card game | Publishers: asmodee, b..."
6,70323,King of Tokyo,0.529660,0.529660,7.07281,61141,Mechanics: card drafting | Publishers: boardga...
7,3076,Puerto Rico,0.522111,0.522111,7.83732,65455,Categories: economic | Mechanics: end game bon...
8,36218,Dominion,0.511559,0.511559,7.49912,81582,Categories: card game | Publishers: filosofia ...
9,266192,Wingspan,0.503945,0.503945,7.94358,56146,Categories: card game | Mechanics: card drafti...


In [26]:
case_trait = discover(
    wanted_categories=["Horror", "Mystery"],
    top_n=10
)

print("=== Case Study: Trait-only ===")
print("Input traits: Horror + Mystery")
display(case_trait[["id", "name", "score_trait", "final_score", "avg_rating", "num_votes", "reason"]].head(10))

=== Case Study: Trait-only ===
Input traits: Horror + Mystery


,id,name,score_trait,final_score,avg_rating,num_votes,reason
0,181304,Mysterium,0.406667,0.487523,7.14588,35262,"Categories: Deduction, Horror | Mechanics: Coo..."
1,127024,Room 25,0.406667,0.438568,6.42697,6131,"Categories: Deduction, Horror | Mechanics: Coo..."
2,181279,Fury of Dracula (Third/Fourth Edition),0.336667,0.428003,7.20338,12479,"Categories: Deduction, Horror | Mechanics: Hid..."
3,253214,Escape Tales: The Awakening,0.406667,0.425923,6.42387,2204,"Categories: Deduction, Horror | Mechanics: Coo..."
4,147949,One Night Ultimate Werewolf,0.336667,0.425753,6.94652,23184,"Categories: Deduction, Horror | Mechanics: Hid..."
5,193037,Dead of Winter: The Long Night,0.336667,0.422867,7.19068,8536,"Categories: Deduction, Horror | Mechanics: Coo..."
6,167355,Nemesis,0.273333,0.417784,7.98226,17710,Categories: Horror | Mechanics: Cooperative Ga...
7,20963,Fury of Dracula (Second Edition),0.336667,0.409359,6.81868,9066,"Categories: Deduction, Horror | Mechanics: Hid..."
8,163166,One Night Ultimate Werewolf: Daybreak,0.336667,0.403821,6.85144,5206,"Categories: Deduction, Horror | Mechanics: Hid..."
9,150376,Dead of Winter: A Crossroads Game,0.266667,0.400911,7.39192,41405,"Categories: Deduction, Horror"


In [27]:
case_combined = discover(
    query_titles=["Splendor", "Azul"],
    wanted_categories=["Strategy", "Family"],
    top_n=10
)

print("=== Case Study: Combined ===")
print("Input titles: Splendor + Azul")
print("Input traits: Strategy + Family")
display(case_combined[["id", "name", "score_like", "score_trait", "final_score", "avg_rating", "num_votes", "reason"]].head(10))

=== Case Study: Combined ===
Input titles: Splendor + Azul
Input traits: Strategy + Family


,id,name,score_like,score_trait,final_score,avg_rating,num_votes,reason
0,822,Carcassonne,0.630833,0.228889,0.623183,7.30857,108776,"Categories: City Building, Territory Building ..."
1,68448,7 Wonders,0.576293,0.273333,0.601113,7.63355,90021,"Categories: card game, economic, City Building..."
2,9209,Ticket to Ride,0.586966,0.140000,0.567450,7.30509,76207,"Mechanics: card drafting, contracts, end game ..."
3,30549,Pandemic,0.592768,0.140000,0.565797,7.48669,109006,"Mechanics: set collection, Hand Management | P..."
4,13,Catan,0.611436,0.044444,0.541287,6.96965,108064,Categories: economic | Mechanics: race | Publi...
5,266192,Wingspan,0.503945,0.184444,0.529174,7.94358,56146,Categories: card game | Mechanics: card drafti...
6,167791,Terraforming Mars,0.473876,0.298889,0.528530,8.27349,74269,"Categories: economic, Territory Building | Mec..."
7,178900,Codenames,0.534498,0.088889,0.507647,7.50765,74456,"Categories: card game, Party Game | Publishers..."
8,3076,Puerto Rico,0.522111,0.088889,0.500383,7.83732,65455,"Categories: economic, City Building | Mechanic..."
9,36218,Dominion,0.511559,0.114444,0.493886,7.49912,81582,Categories: card game | Mechanics: Hand Manage...


In [28]:
def compact_view(df, top_n=10):
    cols = [
        "name", "final_score", "score_like", "score_trait",
        "avg_rating", "num_votes", "reason"
    ]
    cols = [c for c in cols if c in df.columns]

    out = df[cols].copy().head(top_n)

    for c in ["final_score", "score_like", "score_trait", "avg_rating"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round(4)

    if "num_votes" in out.columns:
        out["num_votes"] = pd.to_numeric(out["num_votes"], errors="coerce").fillna(0).astype(int)

    return out

print("=== Compact Title Case ===")
display(compact_view(case_title))

print("=== Compact Trait Case ===")
display(compact_view(case_trait))

print("=== Compact Combined Case ===")
display(compact_view(case_combined))

=== Compact Title Case ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Carcassonne,0.6308,0.6308,0.0,7.3086,108776,Mechanics: tile placement | Publishers: filoso...
1,Catan,0.6114,0.6114,0.0,6.9696,108064,Categories: economic | Mechanics: race | Publi...
2,Pandemic,0.5928,0.5928,0.0,7.4867,109006,Mechanics: set collection | Publishers: asmode...
3,Ticket to Ride,0.5870,0.5870,0.0,7.3051,76207,"Mechanics: card drafting, contracts, end game ..."
4,7 Wonders,0.5763,0.5763,0.0,7.6336,90021,"Categories: card game, economic | Mechanics: s..."
5,Codenames,0.5345,0.5345,0.0,7.5076,74456,"Categories: card game | Publishers: asmodee, b..."
6,King of Tokyo,0.5297,0.5297,0.0,7.0728,61141,Mechanics: card drafting | Publishers: boardga...
7,Puerto Rico,0.5221,0.5221,0.0,7.8373,65455,Categories: economic | Mechanics: end game bon...
8,Dominion,0.5116,0.5116,0.0,7.4991,81582,Categories: card game | Publishers: filosofia ...
9,Wingspan,0.5039,0.5039,0.0,7.9436,56146,Categories: card game | Mechanics: card drafti...


=== Compact Trait Case ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Mysterium,0.4875,0.0,0.4067,7.1459,35262,"Categories: Deduction, Horror | Mechanics: Coo..."
1,Room 25,0.4386,0.0,0.4067,6.4270,6131,"Categories: Deduction, Horror | Mechanics: Coo..."
2,Fury of Dracula (Third/Fourth Edition),0.4280,0.0,0.3367,7.2034,12479,"Categories: Deduction, Horror | Mechanics: Hid..."
3,Escape Tales: The Awakening,0.4259,0.0,0.4067,6.4239,2204,"Categories: Deduction, Horror | Mechanics: Coo..."
4,One Night Ultimate Werewolf,0.4258,0.0,0.3367,6.9465,23184,"Categories: Deduction, Horror | Mechanics: Hid..."
5,Dead of Winter: The Long Night,0.4229,0.0,0.3367,7.1907,8536,"Categories: Deduction, Horror | Mechanics: Coo..."
6,Nemesis,0.4178,0.0,0.2733,7.9823,17710,Categories: Horror | Mechanics: Cooperative Ga...
7,Fury of Dracula (Second Edition),0.4094,0.0,0.3367,6.8187,9066,"Categories: Deduction, Horror | Mechanics: Hid..."
8,One Night Ultimate Werewolf: Daybreak,0.4038,0.0,0.3367,6.8514,5206,"Categories: Deduction, Horror | Mechanics: Hid..."
9,Dead of Winter: A Crossroads Game,0.4009,0.0,0.2667,7.3919,41405,"Categories: Deduction, Horror"


=== Compact Combined Case ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Carcassonne,0.6232,0.6308,0.2289,7.3086,108776,"Categories: City Building, Territory Building ..."
1,7 Wonders,0.6011,0.5763,0.2733,7.6336,90021,"Categories: card game, economic, City Building..."
2,Ticket to Ride,0.5675,0.5870,0.1400,7.3051,76207,"Mechanics: card drafting, contracts, end game ..."
3,Pandemic,0.5658,0.5928,0.1400,7.4867,109006,"Mechanics: set collection, Hand Management | P..."
4,Catan,0.5413,0.6114,0.0444,6.9696,108064,Categories: economic | Mechanics: race | Publi...
5,Wingspan,0.5292,0.5039,0.1844,7.9436,56146,Categories: card game | Mechanics: card drafti...
6,Terraforming Mars,0.5285,0.4739,0.2989,8.2735,74269,"Categories: economic, Territory Building | Mec..."
7,Codenames,0.5076,0.5345,0.0889,7.5076,74456,"Categories: card game, Party Game | Publishers..."
8,Puerto Rico,0.5004,0.5221,0.0889,7.8373,65455,"Categories: economic, City Building | Mechanic..."
9,Dominion,0.4939,0.5116,0.1144,7.4991,81582,Categories: card game | Mechanics: Hand Manage...


In [29]:
print("=== Final Interpretation ===")

best_recall10 = summary_df.sort_values("Recall@10", ascending=False).iloc[0]
best_ndcg10 = summary_df.sort_values("NDCG@10", ascending=False).iloc[0]

print(f"Best model on Recall@10 : {best_recall10['model']} ({best_recall10['Recall@10']:.4f})")
print(f"Best model on NDCG@10   : {best_ndcg10['model']} ({best_ndcg10['NDCG@10']:.4f})")

print("\nSuggested interpretation:")
print("- Title-only is strongest when the user provides clear seed games.")
print("- Trait-only is useful when the user expresses intent without seed titles.")
print("- Combined mode is often the best balance when both taste and intent are available.")
print("- Combined systems are more complex, so their benefit should be justified by measurable gains.")

=== Final Interpretation ===
Best model on Recall@10 : title (0.2033)
Best model on NDCG@10   : title (0.1377)

Suggested interpretation:
- Title-only is strongest when the user provides clear seed games.
- Trait-only is useful when the user expresses intent without seed titles.
- Combined mode is often the best balance when both taste and intent are available.
- Combined systems are more complex, so their benefit should be justified by measurable gains.


In [30]:
print("expand_traits" in globals())
print("TRAIT_EXPANSIONS" in globals())

if "expand_traits" in globals():
    print(expand_traits(["Strategy", "Family"], []))

True
True
(['Strategy', 'Family', 'strategy', 'economic', 'puzzle', 'territory building', 'city building', 'family', "children's game", 'party game', 'card game'], ['hand management', 'tile placement', 'area majority / influence', 'set collection', 'pattern building'])
